# GenAI Pipeline — Grant Scope & Pillar Screening

LLM-based screening of funding grants for alternative protein relevance and pillar classification.
Uses Claude with prompt caching via the Anthropic Python SDK.

Test data: `1_deduplication/data_output/dimensions_data_filtered.xlsx`. Ground-truth `scope` and
`pillar` are derived by matching `Grant ID` against `Identification code` in the `funding_inscope`
table of `funding.db` (grants already tracked by GFI as in-scope) and pulling their `Production
platform` label.

In [1]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

DB_PATH = "funding.db"
XLSX_PATH = "1_deduplication/data_output/dimensions_data_filtered.xlsx"
OUTPUT_DIR = Path("2_scope-pillar")


### 1. Load Test Data

In [2]:
df = pd.read_excel(XLSX_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (173, 40)
Columns: ['Rank', 'Grant ID', 'Grant Number(s)', 'Title', 'Title translated', 'Abstract', 'Abstract translated', 'Keywords', 'Funding amount', 'Currency', 'Funding amount in USD', 'Start date', 'Start Year', 'End Date', 'End Year', 'Researchers', 'Research Organization - original', 'Research Organization - standardized', 'GRID ID', 'City of standardized research organization', 'State of standardized research organization', 'Country of standardized research organization', 'Funder', 'Funder Group', 'Funder Country', 'Program', 'NIH activity code', 'Resulting publications', 'Source Linkout', 'Dimensions URL', 'Fields of Research (ANZSRC 2020)', 'RCDC Categories', 'HRCS HC Categories', 'HRCS RAC Categories', 'Health Research Areas', 'Broad Research Areas', 'Cancer Types', 'CSO Categories', 'Units of Assessment', 'Sustainable Development Goals']


,Rank,Grant ID,Grant Number(s),Title,Title translated,Abstract,Abstract translated,Keywords,Funding amount,Currency,...,Fields of Research (ANZSRC 2020),RCDC Categories,HRCS HC Categories,HRCS RAC Categories,Health Research Areas,Broad Research Areas,Cancer Types,CSO Categories,Units of Assessment,Sustainable Development Goals
0,791,grant.14917108,101182843,Scientific Exchange to assess QUality and Risk...,Scientific Exchange to assess QUality and Risk...,The goal of SEQUR FOOD is to build an internat...,The goal of SEQUR FOOD is to build an internat...,novel foods; bioactive food packaging; shelf l...,1656000,EUR,...,"30 Agricultural, Veterinary and Food Sciences;...",Nutrition,Metabolic and endocrine,3.3 Nutrition and chemoprevention,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",2 Zero Hunger
1,763,grant.15145414,NaN,VegVit,VegVit,"""VegVit"" is een interdisciplinair project dat ...","""VegVit"" is an interdisciplinary project aimed...",fermentation; vitamin B12; vegan alternatives,0,NaN,...,"30 Agricultural, Veterinary and Food Sciences;...",Dietary Supplements; Nutrition,NaN,3.3 Nutrition and chemoprevention,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",NaN
2,740,grant.14631940,NIHR207073,Growing well Study (GWS): the role of portion ...,Growing well Study (GWS): the role of portion ...,What are the study aims? To understand more a...,What are the study aims? To understand more a...,NaN,2418413,GBP,...,32 Biomedical and Clinical Sciences; 3210 Nutr...,Behavioral and Social Science; Clinical Resear...,Cancer; Cardiovascular; Metabolic and endocrin...,"2.3 Psychological, social and economic factors...",Population & Society,Public Health,NaN,NaN,"A03 Allied Health Professions, Dentistry, Nurs...",NaN
3,635,grant.15060106,NaN,"""Een Exclusieve Duurzaamheidstransitie van de ...",An Exclusive Sustainability Transition of Fash...,Dit project neemt de ecologisch destructieve c...,This project takes the ecologically destructiv...,cultural production; polarization; climate soc...,0,NaN,...,"44 Human Society; 4410 Sociology; 47 Language,...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,C21 Sociology,10 Reduced Inequalities
4,5244,grant.15150934,2025-01494_Formas,From Potato Waste to Mycoprotein – A Sustainab...,From Potato Waste to Mycoprotein – A Sustainab...,This project aims to upcycle food-grade potato...,This project aims to upcycle food-grade potato...,NaN,4000000,SEK,...,"30 Agricultural, Veterinary and Food Sciences;...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",12 Responsible Consumption and Production


### 2. Derive Scope and Pillar from `funding_inscope` Match

Compare every `Grant ID` in the test data against `Identification code` in `funding_inscope`. A match means the grant is already tracked by GFI as in-scope. For matched grants, also pull the `Production platform` column as the ground-truth pillar label (Plant-based → PB, Fermentation → F, Cultivated → CM, Cross-cutting → CC). Unmatched (out-of-scope) grants get pillar `NA`.

In [3]:
PLATFORM_TO_PILLAR = {
    "plant-based": "PB",
    "fermentation": "F",
    "cultivated": "CM",
    "cross-cutting": "CC",
}

con = duckdb.connect(DB_PATH, read_only=True)
inscope = con.sql('SELECT "Identification code", "Production platform" FROM funding_inscope').df()
con.close()

inscope = inscope.dropna(subset=["Identification code"]).drop_duplicates(subset="Identification code")
inscope["Identification code"] = inscope["Identification code"].astype(str)

df["grant_id_str"] = df["Grant ID"].astype(str)
df = df.merge(inscope, left_on="grant_id_str", right_on="Identification code", how="left")

df["scope"] = df["Identification code"].notna().map({True: "in", False: "out"})
df["pillar"] = df["Production platform"].str.lower().str.strip().map(PLATFORM_TO_PILLAR).fillna("NA")
df = df.drop(columns=["grant_id_str", "Identification code", "Production platform"])

print(df["scope"].value_counts())
print(f"\n'in' count: {(df['scope'] == 'in').sum()} / {len(df)}")
print(f"\nPillar distribution:\n{df['pillar'].value_counts()}")

scope
out    138
in      35
Name: count, dtype: int64

'in' count: 35 / 173

Pillar distribution:
pillar
NA    138
PB     14
F      12
CC      6
CM      3
Name: count, dtype: int64


### 2b. Apply Manually Corrected Scope Values

Load `scope_corrected` from a previously reviewed results file and overwrite `scope` wherever a correction is present, so subsequent runs are graded against the corrected ground truth.

In [4]:
CORRECTED_SCOPE_PATH = "2_scope-pillar/v1/claude-sonnet-4-6_results_incorrect.xlsx"

corrections = pd.read_excel(CORRECTED_SCOPE_PATH)[["Grant ID", "scope_corrected"]]
corrections = corrections.dropna(subset=["scope_corrected"])

df = df.merge(corrections, on="Grant ID", how="left")
n_corrected = df["scope_corrected"].notna().sum()
df["scope"] = df["scope_corrected"].fillna(df["scope"])
df = df.drop(columns=["scope_corrected"])

print(f"Applied {n_corrected} corrected scope value(s)")
print(df["scope"].value_counts())

Applied 5 corrected scope value(s)
scope
out    143
in      30
Name: count, dtype: int64


### 3. Load Prompt and Select Dataset

No filtering or sampling — the LLM screens the full test set.

In [5]:
PROMPT_PATH = "2_scope-pillar/v3/fund_prompt_v3_conf17_WITHreasoning.md"

DATASET = df
#DATASET = incorrect_scope_data

def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food science research.

Your task is to classify a grant based on its title and abstract. First, determine whether the grant concerns research on alternative proteins intended for use in human food, applying the definitions, boundaries, and out of scope topics below. Provide a confidence score reflecting your certainty in the scope decision. Then, if in scope, assign the relevant alternative protein category label(s).

General Definition
Alternative proteins are plant-based, fermentation-derived, or cultivated substitutes for protein-rich animal-derived foods — including both novel ingredients (e.g. single-cell proteins, mycoprotein) and direct analogues of meat, seafood, dairy, and egg products. Substitution may be at the ingredient level (e.g. a plant protein replacing whey) or the product level (e.g. plant-based dairy replacing conventional dairy), and protein need not be explicitly stated as a goal. Grants are in scope if they fund eith

### 4. API Call with Prompt Caching

In [6]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model
# If choose Opus, comment out TEMPERATURE below below

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESUME_INCOMPLETE = True

# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

class _ClassificationBase(BaseModel):
    scope: Literal["in", "out"]
    confidence: int = Field(ge=1, le=7)
    plant_based: bool
    fermentation: bool
    cultivated: bool
    cross_cutting: bool

class _ClassificationWithReasoning(_ClassificationBase):
    reasoning: str

ClassificationSchema = _ClassificationWithReasoning if INCLUDE_REASONING else _ClassificationBase


client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_grant(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,  #### COMMENT OUT TEMP IF USING OPUS ####
        timeout=REQUEST_TIMEOUT,
        system=[ # Cache control only works for prompts of 1024+ tokens, so not currently useful. Batch API will be later though.
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output


### 5. Error Handling with Retry

In [7]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter

def classify_with_error_handling(row, system_prompt):
    grant_id = row["Grant ID"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_grant(row["Title"], row["Abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {grant_id}: model returned no structured output")
                return {"Grant ID": grant_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["Grant ID"] = grant_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {grant_id}: {last_error}")
    return {"Grant ID": grant_id, "status": "api_error", "error": str(last_error)}


### 6. Checkpoint Helpers

In [8]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["Grant ID"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 7. Run on Test Data

In [9]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["Grant ID"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['Grant ID']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Run 1 / 1
  [1/173] grant.14917108
  [2/173] grant.15145414
  [3/173] grant.14631940
  [4/173] grant.15060106
  [5/173] grant.15150934
  [6/173] grant.14933116
  [7/173] grant.15162658
  [8/173] grant.15166434
  [9/173] grant.14933075
  [10/173] grant.14486646
  [11/173] grant.14878552
  [12/173] grant.15078549
  [13/173] grant.15201530
  [14/173] grant.15195928
  [15/173] grant.15051286
  [16/173] grant.14881068
  [17/173] grant.14950468
  [18/173] grant.14959694
  [19/173] grant.14955094
  [20/173] grant.15196386
  [21/173] grant.14751838
  [22/173] grant.14932889
  Parse failed for grant.14932889: model returned no structured output
  [23/173] grant.15172132
  [24/173] grant.14611344
  [25/173] grant.15197259
  [26/173] grant.15162542
  [27/173] grant.14973726
  [28/173] grant.14881472
  [29/173] grant.14704971
  [30/173] grant.14952019
  [31/173] grant.15201827
  [32/173] grant.14949587
  [33/173] grant.14705292
  [34/173] grant.15147311
  [35/173] grant.14971973
  [36/173] grant.

,scope_LLM,confidence_LLM,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,reasoning_LLM,Grant ID,status,run,error
0,out,2.0,False,False,False,False,Although the project includes oat milk as a ca...,grant.14917108,ok,1,NaN
1,in,7.0,True,True,False,False,This project explicitly develops vegan dairy a...,grant.15145414,ok,1,NaN
2,out,2.0,False,False,False,False,This study examines dietary habits and growth ...,grant.14631940,ok,1,NaN
3,out,1.0,False,False,False,False,This grant concerns the sociology of sustainab...,grant.15060106,ok,1,NaN
4,in,7.0,False,True,False,False,This grant directly concerns biomass fermentat...,grant.15150934,ok,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...
168,out,1.0,False,False,False,False,This grant concerns the molecular biology of s...,grant.15164223,ok,1,NaN
169,out,1.0,False,False,False,False,This grant concerns the modification of potato...,grant.15163833,ok,1,NaN
170,out,1.0,False,False,False,False,This grant is for purchasing a spinning disc c...,grant.15147689,ok,1,NaN
171,out,2.0,False,False,False,False,This grant concerns agronomic and genetic rese...,grant.15163934,ok,1,NaN


Note: there are two routes to receiving a CC label:
1. The LLM labels it as CC. This should only be for very general alt protein grants on socioeconomic type topics e.g. food systems transformation, consumer acceptance etc.
2. The code below (under Derive pillar LLM from boolean flags) determines a CC label applies when more than one individual AP label has been applied by the LLM. This is for more scientific grants that explicitly investigate more than one pillar.

In [10]:
# Compare LLM predictions against ground truth labels
result_cols = ["Grant ID", "run", "scope_LLM", "confidence_LLM", "plant_based_LLM", "fermentation_LLM", "cultivated_LLM", "cross_cutting_LLM", "status", "error"]
if INCLUDE_REASONING:
    result_cols.insert(result_cols.index("confidence_LLM") + 1, "reasoning_LLM")

comparison = DATASET[["Grant ID", "Title", "Abstract", "scope", "pillar"]].merge(
    results_df[result_cols], on="Grant ID", how="left"
)
comparison["correct_scope"] = comparison["scope"] == comparison["scope_LLM"]

# Derive pillar_LLM from boolean flags:
# CC label if multiple pillar flags True OR if both cross_cutting_LLM and no other pillar flags True
# Then, every time only one pillar flag is True, assign that pillar; if none are True, assign NA
pillar_flags = ["plant_based_LLM", "fermentation_LLM", "cultivated_LLM"]
comparison["pillar_LLM"] = comparison[pillar_flags + ["cross_cutting_LLM"]].apply(
    lambda r: "CC" if r[pillar_flags].sum() > 1 or (r["cross_cutting_LLM"] and r[pillar_flags].sum() == 0)
              else "PB" if r["plant_based_LLM"]
              else "F"  if r["fermentation_LLM"]
              else "CM" if r["cultivated_LLM"]
              else "NA",
    axis=1
)
comparison["correct_pillar"] = comparison["pillar"] == comparison["pillar_LLM"]

# Summary metrics
print(f"Scope accuracy ({REPETITIONS} run(s)):  {comparison['correct_scope'].mean():.0%}  (n={len(comparison)})")
print(f"Pillar accuracy ({REPETITIONS} run(s)): {comparison['correct_pillar'].mean():.0%}  (n={len(comparison)})")

both_in = (comparison["scope"] == "in") & (comparison["scope_LLM"] == "in")
print(f"Pillar accuracy — both in-scope:   {comparison.loc[both_in, 'correct_pillar'].mean():.0%}  (n={both_in.sum()})")

if REPETITIONS > 1:
    per_run = comparison.groupby("run")[["correct_scope", "correct_pillar"]].mean()
    print(f"\nPer-run accuracy:\n{per_run.to_string()}")

display_cols = ["Grant ID", "Title", "Abstract", "scope", "scope_LLM", "confidence_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["correct_scope", "pillar", "pillar_LLM", "correct_pillar", "plant_based_LLM", "fermentation_LLM", "cultivated_LLM", "cross_cutting_LLM", "status", "error"]
comparison[display_cols]


Scope accuracy (1 run(s)):  92%  (n=173)
Pillar accuracy (1 run(s)): 86%  (n=173)
Pillar accuracy — both in-scope:   74%  (n=23)


,Grant ID,Title,Abstract,scope,scope_LLM,confidence_LLM,reasoning_LLM,correct_scope,pillar,pillar_LLM,correct_pillar,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,status,error
0,grant.14917108,Scientific Exchange to assess QUality and Risk...,The goal of SEQUR FOOD is to build an internat...,out,out,2.0,Although the project includes oat milk as a ca...,True,NA,NA,True,False,False,False,False,ok,NaN
1,grant.15145414,VegVit,"""VegVit"" is een interdisciplinair project dat ...",in,in,7.0,This project explicitly develops vegan dairy a...,True,PB,CC,False,True,True,False,False,ok,NaN
2,grant.14631940,Growing well Study (GWS): the role of portion ...,What are the study aims? To understand more a...,out,out,2.0,This study examines dietary habits and growth ...,True,NA,NA,True,False,False,False,False,ok,NaN
3,grant.15060106,"""Een Exclusieve Duurzaamheidstransitie van de ...",Dit project neemt de ecologisch destructieve c...,out,out,1.0,This grant concerns the sociology of sustainab...,True,NA,NA,True,False,False,False,False,ok,NaN
4,grant.15150934,From Potato Waste to Mycoprotein – A Sustainab...,This project aims to upcycle food-grade potato...,in,in,7.0,This grant directly concerns biomass fermentat...,True,F,F,True,False,True,False,False,ok,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,grant.15164223,Rola długich niekodujących RNA (lncRNA) związa...,Z chemicznego punktu widzenia DNA jest prostym...,out,out,1.0,This grant concerns the molecular biology of s...,True,NA,NA,True,False,False,False,False,ok,NaN
169,grant.15163833,Związki fenolowe jako modyfikatory funkcjonaln...,Celem projektu jest poznanie wpływu wybranych ...,out,out,1.0,This grant concerns the modification of potato...,True,NA,NA,True,False,False,False,False,ok,NaN
170,grant.15147689,A broadly accessible and versatile spinning di...,"In this proposal, we request funds to purchase...",out,out,1.0,This grant is for purchasing a spinning disc c...,True,NA,NA,True,False,False,False,False,ok,NaN
171,grant.15163934,Geny warunkujące neutralność fotoperiodyczną i...,Głównym celem projektu jest identyfikacja kluc...,out,out,2.0,This grant concerns agronomic and genetic rese...,True,NA,NA,True,False,False,False,False,ok,NaN


### 8. Save Results to Excel

In [ ]:
save_dir = OUTPUT_DIR / "v3"  # CHANGE ME as prompt versions iterate
save_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([
    {"metric": "scope_accuracy",                "value": f"{comparison['correct_scope'].mean():.0%}",               "n": len(comparison)},
    {"metric": "pillar_accuracy",               "value": f"{comparison['correct_pillar'].mean():.0%}",              "n": len(comparison)},
    {"metric": "pillar_accuracy_both_in_scope", "value": f"{comparison.loc[both_in, 'correct_pillar'].mean():.0%}", "n": int(both_in.sum())},
])

with pd.ExcelWriter(save_dir / f"{MODEL}_results.xlsx") as writer:
    comparison[display_cols].to_excel(writer, sheet_name="comparison", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)

print(f"Saved to {save_dir / f'{MODEL}_results.xlsx'}")


Saved to 2_scope-pillar\v2\claude-sonnet-4-6_results.xlsx


Create dataset of only rows where the LLM got scope wrong, for re-run with a modified prompt.

In [23]:
incorrect_ids = comparison.loc[~comparison["correct_scope"], "Grant ID"]
incorrect_scope_data = DATASET[DATASET["Grant ID"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_scope_data


,Rank,Grant ID,Grant Number(s),Title,Title translated,Abstract,Abstract translated,Keywords,Funding amount,Currency,...,HRCS HC Categories,HRCS RAC Categories,Health Research Areas,Broad Research Areas,Cancer Types,CSO Categories,Units of Assessment,Sustainable Development Goals,scope,pillar
0,791,grant.14917108,101182843,Scientific Exchange to assess QUality and Risk...,Scientific Exchange to assess QUality and Risk...,The goal of SEQUR FOOD is to build an internat...,The goal of SEQUR FOOD is to build an internat...,novel foods; bioactive food packaging; shelf l...,1656000,EUR,...,Metabolic and endocrine,3.3 Nutrition and chemoprevention,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",2 Zero Hunger,out,NA
1,1862,grant.14878552,LS24-058,Designing a professional-secretory synthetic e...,Designing a professional-secretory synthetic e...,The endoplasmic reticulum (ER) is the key orga...,The endoplasmic reticulum (ER) is the key orga...,NaN,897906,EUR,...,Generic health relevance,NaN,NaN,Basic Science,NaN,NaN,A05 Biological Sciences,NaN,in,CC
2,1269,grant.14932889,271467; 101209784; Tailored Immunity; 10.3030/...,Tailored immune receptor engineering for resis...,Tailored immune receptor engineering for resis...,Fungal pathogens are a major constraint to cro...,Fungal pathogens are a major constraint to cro...,NaN,242261,EUR,...,Infection,2.1 Biological and endogenous factors,Biomedical,Basic Science,NaN,NaN,A05 Biological Sciences,NaN,out,NA
3,1097,grant.15162542,PCI2025-167111-2,MICROALGAL BIOMASS VALORIZATION FOR SUSTAINABL...,MICROALGAL BIOMASS VALORIZATION FOR SUSTAINABL...,EL PROYECTO BIOVAL RESPONDE AL CRECIENTE DESAF...,The BIOVAL project responds to the growing cha...,NaN,212500,EUR,...,NaN,NaN,NaN,NaN,NaN,NaN,B12 Engineering,12 Responsible Consumption and Production,in,F
4,1064,grant.14704971,25-23-20221,Development of approaches to complex processin...,Development of approaches to complex processin...,The proposed study is devoted to the developme...,The proposed study is devoted to the developme...,Microalgae; cultivation; antioxidant activity;...,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,B12 Engineering,NaN,in,F
5,871,grant.15201374,UKRI2987,Novel Mechanism-of-Action (NMoA) genes: a path...,Novel Mechanism-of-Action (NMoA) genes: a path...,Implementing the latest knowledge of plant sci...,Implementing the latest knowledge of plant sci...,NaN,724875,GBP,...,NaN,NaN,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",2 Zero Hunger,out,NA
6,840,grant.15147565,UKRI1917,BacCORAC: Bacterial modulation of plant respon...,BacCORAC: Bacterial modulation of plant respon...,"Potato, the world’s third main food crop, is p...","Potato, the world’s third main food crop, is p...",NaN,883292,GBP,...,NaN,NaN,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",2 Zero Hunger,out,NA
7,830,grant.15148367,UKRI1913,"iTKP fusions: Structure, function and engineer...","iTKP fusions: Structure, function and engineer...",Context and Challenge: Yield from key crops th...,Context and Challenge: Yield from key crops th...,NaN,522180,GBP,...,Infection,NaN,Biomedical,Basic Science,NaN,NaN,A05 Biological Sciences,2 Zero Hunger,out,NA
8,806,grant.14954840,101180167; 272628; PRIMARY; 10.3030/101180167,New business for farmers and cooperatives in r...,New business for farmers and cooperatives in r...,PRIMARY aims to develop and pilot local soluti...,PRIMARY aims to develop and pilot local soluti...,NaN,4996936,EUR,...,NaN,NaN,NaN,NaN,NaN,NaN,B12 Engineering,2 Zero Hunger,in,F
9,724,grant.14487762,101126022; 267984; SynUbL; 10.3030/101126022,Improving plant immunity by synthetic exploita...,Improving plant immunity by synthetic exploita...,Plants are continuously attacked by pathogens ...,Plants are continuously attacked by pathogens ...,NaN,1850374,EUR,...,Infection,2.1 Biological and endogenous factors,Biomedical,Basic Science,NaN,NaN,A05 Biological Sciences,2 Zero Hunger,out,NA
